# 🧠 SNAP-C1 Training Pipeline (Google Colab T4)
**Self-Neural Adaptive Processing - Core 1**

This notebook trains all 3 SNAP-C1 LoRA adapters on Qwen3-8B using QLoRA (4-bit),
then merges them and exports to GGUF for local inference on LM Studio (AMD RX 7600).

**Runtime**: T4 GPU (16GB VRAM) | **Time**: ~2-3 hours total

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q transformers>=4.45.0 accelerate>=0.34.0 peft>=0.13.0
!pip install -q bitsandbytes>=0.44.0 trl>=0.12.0 datasets>=3.0.0
!pip install -q pyyaml loguru tqdm rich
!pip install -q llama-cpp-python  # For GGUF export

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 📂 Step 2: Clone the SNAP-C1 Repository

In [ ]:
!git clone https://github.com/IRSPlays/SNAP-C1.git
%cd SNAP-C1
!ls

## 🔧 Step 3: Setup Model & Quantization Config
Configure Qwen3-8B with 4-bit QLoRA for the T4 GPU.

In [ ]:
import torch
import yaml
import json
import os
from pathlib import Path
from loguru import logger
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig

PROJECT_ROOT = Path(".")
CONFIG_DIR = PROJECT_ROOT / "config"

# ---- Model Configuration ----
MODEL_NAME = "Qwen/Qwen3-8B"

# 4-bit quantization config for T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # T4 supports bf16
    bnb_4bit_use_double_quant=True,
)

print(f"Model: {MODEL_NAME}")
print(f"Quantization: 4-bit NF4 with double quantization")
print(f"Compute dtype: bfloat16")

## 🧠 Step 4: Load Base Model

In [ ]:
logger.info(f"Loading {MODEL_NAME} in 4-bit...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

logger.info(f"Model loaded. Parameters: {model.num_parameters():,}")
if torch.cuda.is_available():
    free_vram = torch.cuda.mem_get_info()[0] / 1024**3
    logger.info(f"Free VRAM: {free_vram:.1f} GB")

## 📊 Step 5: Training Helper Functions

In [ ]:
def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def format_chat(example, tokenizer):
    """Format training example into Qwen3 chat template."""
    messages = [
        {"role": "system", "content": (
            "You are SNAP-C1 (Self-Neural Adaptive Processing - Core 1), "
            "an advanced AI that reasons from multiple perspectives, "
            "self-corrects its outputs, and uses tools when needed. "
            "Think deeply before responding."
        )},
    ]
    instruction = example['instruction']
    if 'initial_response' in example:
        instruction += f"\n\nPrevious attempt:\n{example['initial_response']}\n\nPlease review and correct this response."
    messages.append({"role": "user", "content": instruction})
    messages.append({"role": "assistant", "content": example['output']})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def train_adapter(adapter_config_path, base_model, tokenizer):
    """Train a single LoRA adapter."""
    config = load_yaml(adapter_config_path)
    adapter_name = config['adapter']['name']
    logger.info(f"\n{'='*60}")
    logger.info(f"Training adapter: {adapter_name}")
    logger.info(f"Description: {config['adapter']['description']}")
    logger.info(f"{'='*60}")

    # LoRA config
    lora_cfg = config['lora']
    peft_config = LoraConfig(
        r=lora_cfg['r'],
        lora_alpha=lora_cfg['lora_alpha'],
        lora_dropout=lora_cfg['lora_dropout'],
        target_modules=lora_cfg['target_modules'],
        bias=lora_cfg['bias'],
        task_type=TaskType.CAUSAL_LM,
    )
    peft_model = get_peft_model(base_model, peft_config)
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    logger.info(f"Trainable params: {trainable:,}")

    # Load dataset
    data_cfg = config['data']
    data_path = PROJECT_ROOT / data_cfg['path']
    train_data = load_jsonl(data_path / data_cfg['train_split'])
    logger.info(f"Training examples: {len(train_data)}")

    eval_data = None
    eval_path = data_path / data_cfg.get('eval_split', 'eval.jsonl')
    if eval_path.exists():
        eval_data = load_jsonl(eval_path)
        logger.info(f"Eval examples: {len(eval_data)}")

    train_dataset = Dataset.from_list(train_data).map(lambda x: {'text': format_chat(x, tokenizer)})
    eval_dataset = Dataset.from_list(eval_data).map(lambda x: {'text': format_chat(x, tokenizer)}) if eval_data else None

    # Training args — optimized for T4
    t = config['training']
    output_dir = PROJECT_ROOT / 'adapters' / adapter_name

    training_args = SFTConfig(
        output_dir=str(output_dir),
        num_train_epochs=t['num_epochs'],
        per_device_train_batch_size=t['batch_size'],
        gradient_accumulation_steps=t['gradient_accumulation_steps'],
        learning_rate=t['learning_rate'],
        lr_scheduler_type=t['lr_scheduler'],
        warmup_ratio=t['warmup_ratio'],
        weight_decay=t['weight_decay'],
        max_length=t['max_seq_length'],
        fp16=False,
        bf16=True,  # T4 supports bf16
        optim='paged_adamw_8bit',
        logging_steps=t['logging_steps'],
        save_steps=t['save_steps'],
        save_total_limit=2,
        seed=t['seed'],
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        report_to='none',
        dataset_text_field='text',
        packing=False,
        eval_strategy='steps' if eval_dataset else 'no',
        eval_steps=t['save_steps'] if eval_dataset else None,
    )

    trainer = SFTTrainer(
        model=peft_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )

    logger.info("Starting training...")
    trainer.train()

    # Save adapter
    final_path = output_dir / 'final'
    peft_model.save_pretrained(str(final_path))
    tokenizer.save_pretrained(str(final_path))
    logger.info(f"Adapter saved: {final_path}")

    # Save metrics
    with open(output_dir / 'training_metrics.json', 'w') as f:
        json.dump(trainer.state.log_history, f, indent=2)

    # Cleanup for next adapter
    peft_model.unload()
    torch.cuda.empty_cache()

    logger.info(f"✅ {adapter_name} training complete!")
    return str(final_path)

print("Helper functions loaded ✅")

## 🚀 Step 6: Train All 3 Adapters
This trains team_thinking, self_correction, and tool_use sequentially.

In [ ]:
adapters_to_train = [
    "config/team_thinking.yaml",
    "config/self_correction.yaml",
    "config/tool_use.yaml",
]

trained_paths = []
for config_path in adapters_to_train:
    if Path(config_path).exists():
        path = train_adapter(config_path, model, tokenizer)
        trained_paths.append(path)
    else:
        logger.warning(f"Config not found: {config_path}")

print(f"\n🎉 All adapters trained!")
for p in trained_paths:
    print(f"  📁 {p}")

## 🔀 Step 7: Merge Adapters into Base Model
Merge the trained LoRA weights permanently into the base Qwen3-8B model.

In [ ]:
# Reload the base model in fp16 for merging
logger.info("Reloading base model in fp16 for adapter merging...")

merge_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

# Load and merge each trained adapter
adapter_names = ['team_thinking', 'self_correction', 'tool_use']
for adapter_name in adapter_names:
    adapter_path = PROJECT_ROOT / 'adapters' / adapter_name / 'final'
    if adapter_path.exists():
        logger.info(f"Merging adapter: {adapter_name}")
        merge_model = PeftModel.from_pretrained(merge_model, str(adapter_path))
        merge_model = merge_model.merge_and_unload()
        logger.info(f"✅ {adapter_name} merged")
    else:
        logger.warning(f"Adapter not found: {adapter_path}")

# Save the merged model
merged_path = PROJECT_ROOT / 'models' / 'snap-c1-merged'
merged_path.mkdir(parents=True, exist_ok=True)
merge_model.save_pretrained(str(merged_path))
tokenizer.save_pretrained(str(merged_path))
logger.info(f"\n🎉 Merged model saved to: {merged_path}")

del merge_model
torch.cuda.empty_cache()

## 📦 Step 8: Export to GGUF
Convert the merged model to GGUF format (Q4_K_M) for LM Studio on your RX 7600.

In [ ]:
# Install llama.cpp conversion tools
!pip install -q gguf sentencepiece
!git clone https://github.com/ggerganov/llama.cpp.git /content/llama.cpp

# Convert HuggingFace model to GGUF
!python /content/llama.cpp/convert_hf_to_gguf.py \
    models/snap-c1-merged \
    --outfile models/snap-c1-qwen3-8b-f16.gguf \
    --outtype f16

print("\n✅ GGUF f16 conversion complete!")

In [ ]:
# Quantize to Q4_K_M (fits in 8GB VRAM on RX 7600)
# First, build llama.cpp quantize tool
!cd /content/llama.cpp && cmake -B build && cmake --build build --config Release -j$(nproc) --target llama-quantize

!cd /content/SNAP-C1 && /content/llama.cpp/build/bin/llama-quantize \
    models/snap-c1-qwen3-8b-f16.gguf \
    models/snap-c1-qwen3-8b-Q4_K_M.gguf \
    Q4_K_M

import os
gguf_path = 'models/snap-c1-qwen3-8b-Q4_K_M.gguf'
size_gb = os.path.getsize(gguf_path) / 1024**3
print(f"\n🎉 Final SNAP-C1 GGUF model: {gguf_path}")
print(f"📏 Size: {size_gb:.1f} GB")
print(f"🖥️ Ready for LM Studio on AMD RX 7600!")

## 💾 Step 9: Download Your SNAP-C1 Model
Download the GGUF file to your local machine, then load it in LM Studio.

In [ ]:
from google.colab import files

print("Downloading SNAP-C1 GGUF model...")
print("This may take a few minutes depending on your internet speed.")
files.download('models/snap-c1-qwen3-8b-Q4_K_M.gguf')

print("\n" + "="*60)
print("🎉 SNAP-C1 Training Complete!")
print("="*60)
print("\nNext steps:")
print("1. Open LM Studio on your PC")
print("2. Click 'My Models' -> Import")
print("3. Select the downloaded snap-c1-qwen3-8b-Q4_K_M.gguf")
print("4. Load the model and start the local server")
print("5. SNAP-C1 is now running on your AMD RX 7600! 🚀")